## Baseline model

In [52]:
from string import digits

import numpy as np
import pandas as pd

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, train_test_split, cross_val_predict
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, f1_score, roc_auc_score, average_precision_score
import warnings

import matplotlib.pyplot as plt
import seaborn as sns


In [43]:
RANDOM_STATE = 42
%matplotlib inline
warnings.filterwarnings("ignore")

In [44]:
df = pd.read_csv('../../data/preprocessed.csv')
df.sample(5)

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
131352,774,Spain,Male,50,2,0.00,2,1,1,129964.50,0
159828,679,France,Male,34,6,0.00,2,1,0,176868.43,0
69869,686,Spain,Male,28,3,122931.55,1,1,1,73429.38,0
78616,775,France,Male,54,4,0.00,2,1,0,88721.84,0
46984,745,France,Male,62,3,0.00,1,0,1,161519.76,1


### 1. Линейные модели классификации

#### 1.1 Кодирование категориальных признаков

к колонкам `Gender`, `Geography` с малым количеством категорий применим в пайплайне One-Hot-Encoding

In [45]:
print(df['Gender'].value_counts())
print(df['Geography'].value_counts())

X, y = df.drop(columns='Exited'), df['Exited']
print(X.describe())

num_cols = ['CreditScore', 'Age', 'Balance', 'EstimatedSalary', 'Tenure']
ohe_features = ['Gender', 'Geography']
ohe = OneHotEncoder(drop='first', handle_unknown='ignore')

Gender
Male      93150
Female    71884
Name: count, dtype: int64
Geography
France     94215
Spain      36213
Germany    34606
Name: count, dtype: int64
         CreditScore            Age         Tenure        Balance  \
count  165034.000000  165034.000000  165034.000000  165034.000000   
mean      656.454373      38.125883       5.020353   55478.086689   
std        80.103340       8.867207       2.806159   62817.663278   
min       350.000000      18.000000       0.000000       0.000000   
25%       597.000000      32.000000       3.000000       0.000000   
50%       659.000000      37.000000       5.000000       0.000000   
75%       710.000000      42.000000       7.000000  119939.517500   
max       850.000000      92.000000      10.000000  250898.090000   

       NumOfProducts      HasCrCard  IsActiveMember  EstimatedSalary  
count  165034.000000  165034.000000   165034.000000    165034.000000  
mean        1.554455       0.753954        0.497770    112574.822734  
std         0

In [46]:
preprocessor = ColumnTransformer(
    transformers=[
        ('categ', ohe, ohe_features),
        ('num', StandardScaler(), num_cols)
    ], remainder='passthrough'
)

#### 1.2 Пайплайн обучения модели с учетом дисбаланса классов

In [47]:
print(f"Баланс классов: {y.value_counts(normalize=True)}")
# для честной оценки модели: не устраняет дисбаланс, а сохраняет его пропорцию в каждом фолде
skf = StratifiedKFold(n_splits=4, shuffle=True, random_state=RANDOM_STATE)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', LogisticRegression(max_iter=1000,
                               random_state=RANDOM_STATE,
                               class_weight='balanced'  # автоматически взвешивать классы
                               ))
])
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=RANDOM_STATE, stratify=y)
pipeline.fit(X_train, y_train)

Баланс классов: Exited
0    0.788401
1    0.211599
Name: proportion, dtype: float64


,steps,"[('preprocessor', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('categ', ...), ('num', ...)]"
,remainder,'passthrough'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


#### 1.3 Выбор метрики качества моделей

Accuracy (доля верных ответов) - очень плохой выбор для оценки моделей при дисбалансе классов. 
Рассмотрим другие метрики. Будем учитывать, что в нашей бизнес-задаче важнее не допустить оттока клиентов (class 1 - доля в таргете 21%).

* Precision (точность) - какая доля из помеченных уходящими действительно уйдёт. Второстепенная метрика 
* Recall (полнота) - какую долю модель выявила из реально уходящих. Важная метрика
* F-1 мера - усредненная метрика по точности и полноте. Подходит для несбалансированных данных
* ROC-AUC - интегральная метрика, не требует подбора порога для каждой модели. Хороша для подбора модели
* PR-AUC - также интегральная метрика. При сильном дисбалансе более информативна, показывает, насколько хорошо модель находит редкий класс

##### Итоги: 
При подборе гиперпараметров в `GridSearch` будем использовать `recall`, но видеть и PR-AUC(scoring=`average_precision`). В classification_report обращаем внимание на 1 класс.

In [50]:
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)

print(f"\nPR-AUC: {average_precision_score(y_test, y_proba):.4f}")

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.76      0.83     32529
           1       0.45      0.73      0.56      8730

    accuracy                           0.75     41259
   macro avg       0.68      0.75      0.69     41259
weighted avg       0.82      0.75      0.77     41259


Confusion Matrix:
[[24689  7840]
 [ 2321  6409]]

PR-AUC: 0.5706
F1 (class 1): 0.5578


#### 1.4 Подбор гиперпараметров модели логистической регрессии

In [54]:
param_grid = {
    'clf__C': np.logspace(-2, 2, 7),
    'clf__class_weight': ['balanced', {0: 1, 1: 4}, {0: 1, 1: 6}]
}
grid = GridSearchCV(pipeline,
                    param_grid,
                    cv=skf,
                    scoring={
                        'pr_auc': 'average_precision',
                        'recall': 'recall'  # Recall (по порогу 0.5)
                    }, 
                    refit='recall',
                    n_jobs=-1,
                    return_train_score=True)
grid.fit(X_train, y_train)

,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'clf__C': array([1.0000...00000000e+02]), 'clf__class_weight': ['balanced', {0: 1, 1: 4}, ...]}"
,scoring,"{'pr_auc': 'average_precision', 'recall': 'recall'}"
,n_jobs,-1
,refit,'recall'
,cv,StratifiedKFo... shuffle=True)
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,True
,transformers,"[('categ', ...), ('num', ...)]"


In [58]:
best_model = grid.best_estimator_

# Предсказания на тесте
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print(f"PR-AUC: {average_precision_score(y_test, y_proba):.4f}")

# 4. Итоги поиска
print(f"\nЛучшие параметры: {grid.best_params_}")
print(f"Лучший Recall (по CV): {grid.best_score_:.4f}")

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.62      0.75     32529
           1       0.37      0.83      0.51      8730

    accuracy                           0.67     41259
   macro avg       0.65      0.73      0.63     41259
weighted avg       0.81      0.67      0.70     41259


Confusion Matrix:
[[20250 12279]
 [ 1459  7271]]
PR-AUC: 0.5664

Лучшие параметры: {'clf__C': np.float64(0.01), 'clf__class_weight': {0: 1, 1: 6}}
Лучший Recall (по CV): 0.8354
